# AVA Drone Agent 

In [ ]:
!pip install -q agno dronekit pymavlink future pyyaml pydantic

In [ ]:
import os

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = input('OpenRouter API Key: ')

print('API key set.' if os.environ.get('OPENROUTER_API_KEY') else 'No API key!')

In [ ]:
from dronekit import connect, VehicleMode, LocationGlobal, LocationGlobalRelative
from pymavlink import mavutil
import time

CONNECTION_STRING = 'tcp:127.0.0.1:5763'

print('Connecting to vehicle...')
vehicle = connect(CONNECTION_STRING, wait_ready=True, baud=57600, rate=60)
print(f'Connected!')
print(f'  Mode:     {vehicle.mode.name}')
print(f'  Armed:    {vehicle.armed}')
print(f'  Battery:  {vehicle.battery}')
print(f'  GPS:      {vehicle.gps_0}')
print(f'  Alt:      {vehicle.location.global_relative_frame.alt}m')

# Define Drone Tools

Agno automatically:
- Reads the function signature and docstring
- Creates the tool schema for the LLM
- Calls the function when the LLM invokes the tool


In [ ]:
import math
from agno.tools import tool


@tool
def set_mode(mode: str) -> str:
    """
    Set the vehicle flight mode.
    Available modes: GUIDED, ALT_HOLD, RTL, AUTO, STABILIZE, LAND
    Args:
        mode: The flight mode to set (e.g. 'GUIDED', 'RTL')
    """
    vehicle.mode = VehicleMode(mode)
    time.sleep(1)
    return f'Mode set to {vehicle.mode.name}'


@tool
def arm_vehicle(arm: bool) -> str:
    """
    Arm or disarm the vehicle motors.
    Args:
        arm: True to arm, False to disarm
    """
    vehicle.armed = arm
    timeout = 10
    while vehicle.armed != arm and timeout > 0:
        time.sleep(1)
        timeout -= 1
    status = 'armed' if vehicle.armed else 'disarmed'
    return f'Vehicle is now {status}'


@tool
def takeoff(alt: float) -> str:
    """
    Take off to a specific altitude in meters.
    Default altitude is 10 meters if not specified.
    Vehicle must be armed and in GUIDED mode first.
    Args:
        alt: Target altitude in meters
    """
    if not vehicle.armed:
        return 'ERROR: Vehicle must be armed before takeoff'
    if vehicle.mode.name != 'GUIDED':
        return 'ERROR: Vehicle must be in GUIDED mode for takeoff'
    
    vehicle.simple_takeoff(alt)
    # Wait until we reach ~95% of target altitude
    while vehicle.location.global_relative_frame.alt < alt * 0.95:
        current = round(vehicle.location.global_relative_frame.alt, 1)
        print(f'  Climbing... {current}m / {alt}m')
        time.sleep(1)
    return f'Takeoff complete. Altitude: {round(vehicle.location.global_relative_frame.alt, 1)}m'


@tool
def go_to_coords(lat: float, lon: float, alt: float = None, frame: str = 'Relative') -> str:
    """
    Fly to GPS coordinates (latitude, longitude, optional altitude).
    Args:
        lat: Destination latitude in decimal degrees
        lon: Destination longitude in decimal degrees
        alt: Destination altitude in meters. If not provided, maintains current altitude.
        frame: Reference frame - 'Relative' (above home) or 'Global' (absolute). Default: Relative
    """
    if alt is None:
        alt = vehicle.location.global_relative_frame.alt
    
    if frame == 'Relative':
        dest = LocationGlobalRelative(lat, lon, alt)
    else:
        dest = LocationGlobal(lat, lon, alt)
    
    vehicle.simple_goto(dest)
    return f'Navigating to lat={lat}, lon={lon}, alt={alt}m ({frame} frame)'


@tool
def go_to_local(x: float, y: float, z: float) -> str:
    """
    Move relative to current position using local frame (Forward/Right/Up).
    Limited to 10 meters per axis for safety.
    Args:
        x: Forward (+) / Backward (-) in meters
        y: Right (+) / Left (-) in meters
        z: Up (+) / Down (-) in meters
    """
    # Clamp values to +/-10m
    x = max(-10, min(10, x))
    y = max(-10, min(10, y))
    z = max(-10, min(10, z))
    
    current_lat = vehicle.location.global_relative_frame.lat
    current_lon = vehicle.location.global_relative_frame.lon
    current_alt = vehicle.location.global_relative_frame.alt
    
    # Approximate conversion: 1 degree lat ~ 111320m, 1 degree lon varies with lat
    yaw = vehicle.attitude.yaw  # radians
    dx_lat = (x * math.cos(yaw) - y * math.sin(yaw)) / 111320.0
    dx_lon = (x * math.sin(yaw) + y * math.cos(yaw)) / (111320.0 * math.cos(math.radians(current_lat)))
    
    dest = LocationGlobalRelative(
        current_lat + dx_lat,
        current_lon + dx_lon,
        current_alt + z
    )
    vehicle.simple_goto(dest)
    return f'Moving: forward={x}m, right={y}m, up={z}m'


@tool
def set_heading(yaw: float, frame: str = 'Relative') -> str:
    """
    Set the vehicle heading/yaw angle.
    Args:
        yaw: Target yaw angle in degrees.
             Relative frame: -180 to 180 (positive = clockwise, negative = counterclockwise)
             Global frame: 0 to 360 (0=North, 90=East, 180=South, 270=West)
        frame: 'Relative' (turn relative to current heading) or 'Global' (absolute compass heading). Default: Relative
    """
    is_relative = 1 if frame == 'Relative' else 0
    direction = 1  # 1=clockwise, -1=ccw
    
    msg = vehicle.message_factory.command_long_encode(
        0, 0,
        mavutil.mavlink.MAV_CMD_CONDITION_YAW, 0,
        abs(yaw),        # yaw angle
        0,               # yaw speed deg/s (0 = default)
        direction if yaw >= 0 else -1,  # direction
        is_relative,     # relative or absolute
        0, 0, 0
    )
    vehicle.send_mavlink(msg)
    return f'Heading set to {yaw} degrees ({frame})'


@tool
def get_vehicle_status() -> str:
    """
    Get current vehicle telemetry status.
    Returns mode, armed state, battery, altitude, position, speed, and heading.
    Use this when the user asks about the drone's current state.
    """
    heading_deg = round(math.degrees(vehicle.attitude.yaw), 1)
    return (
        f'Mode: {vehicle.mode.name} | '
        f'Armed: {vehicle.armed} | '
        f'Battery: {vehicle.battery.voltage}V ({vehicle.battery.level}%) | '
        f'Alt: {round(vehicle.location.global_relative_frame.alt, 1)}m | '
        f'Lat: {vehicle.location.global_relative_frame.lat} | '
        f'Lon: {vehicle.location.global_relative_frame.lon} | '
        f'Groundspeed: {round(vehicle.groundspeed, 1)} m/s | '
        f'Heading: {heading_deg} deg'
    )


# Collect all tools for the agent
drone_tools = [set_mode, arm_vehicle, takeoff, go_to_coords, go_to_local, set_heading, get_vehicle_status]
print(f'Defined {len(drone_tools)} drone tools:')
for t in drone_tools:
    print(f'  - {t.name}')

Agno handles:
- **Tool discovery** — reads `@tool` function signatures and docstrings automatically
- **Tool execution** — when the LLM returns a tool call, Agno runs the function
- **Session memory** — `InMemoryDb` + `add_history_to_context` gives the agent conversation history
- **Structured output** — optional Pydantic schema for typed responses


In [ ]:
from textwrap import dedent
from agno.agent import Agent
from agno.models.openrouter import OpenRouter
from agno.db.in_memory import InMemoryDb

# Agent memory
db = InMemoryDb()
SESSION_ID = 'drone_session_001'

# System prompt 
SYSTEM_PROMPT = dedent("""
    You are an AI copilot for drone operations.
    Speak ultra-concisely with an operational tone.
    
    You have tools to control a drone connected via DroneKit to a Mission Planner SITL.
    
    Rules:
    - For flight commands: use the appropriate tool(s)
    - For questions about drone state: use get_vehicle_status
    - For general questions: respond from your knowledge about drone operations
    - If a command requires multiple steps (e.g. 'take off' implies arm + mode + takeoff),
      execute them in the correct sequence
    - Always confirm what you did after executing commands
""")

drone_agent = Agent(
    name='AVA_DroneAgent',
    model=OpenRouter(
        id='qwen/qwen3-vl-235b-a22b-instruct',  
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
    instructions=SYSTEM_PROMPT,
    tools=drone_tools,
    db=db,
    session_id=SESSION_ID,
    add_history_to_context=True,   # Agent remembers previous turns
    read_tool_call_history=True,    # Agent can see what tools it already called
    retries=3,
    debug_mode=True,               # Set to False for cleaner output
)

print('Agent ready.')
print(f'  Model:   {drone_agent.model.id}')
print(f'  Tools:   {[t.name for t in drone_tools]}')
print(f'  Session: {SESSION_ID}')

In [ ]:
# Quick test — ask about vehicle status (doesn't move anything)
result = drone_agent.run('What is the current status of the drone?')
print('\nAgent Response:', result.content)

##  Interactive Command Loop


In [ ]:
print('=== AVA Drone Agent — Interactive Mode ===')
print('Type commands in natural language. Type "exit" to quit.\n')

while True:
    user_input = input('\nCommand: ')
    
    if user_input.lower().strip() == 'exit':
        print('Closing...')
        break
    
    if not user_input.strip():
        continue
    
    start_time = time.time()
    
    try:
        result = drone_agent.run(user_input)
        latency = time.time() - start_time
        
        print(f'\n  Response: {result.content}')
        print(f'  Latency:  {latency:.2f}s')
    except Exception as e:
        print(f'\n  ERROR: {e}')

print('\nSession ended.')

Close the DroneKit connection when done.

In [ ]:
vehicle.close()
print('Vehicle connection closed.')